# HumanAI Detect - Kalibrasyon Olcumu (after, kisa-pilot verisiyle) (Colab)

**Amac:** Yerel makinede `scripts/measure_calibration.py --stage after --save-models`
cok yavas gidiyor (4 base model x 5-fold kalibrasyon + stacking'in kendi 5-fold'u,
9479 ornek x 803 ozellik). Bu notebook ayni islemi Colab'da calistirir.

**GPU runtime secili gelir** (Runtime > Change runtime type > T4 GPU zaten
ayarli). Not: XGBoost/CatBoost bu projede GPU'ya ozel yapilandirilmadi (CPU
backend), o yuzden GPU'nun hiz katkisi sinirli olabilir -- asil kazanc Colab'in
kendi CPU/RAM kapasitesinin ve kesintisiz calisma ortaminin yerel makineye gore
daha guvenilir olmasi. Sonuclarin yerel calistirmayla birebir karsilastirilabilir
kalmasi icin model backend'i BILEREK degistirilmedi.

**Girdi:** `data/processed/fused.parquet` + `data/processed/holdout_ids.txt` --
bunlar yerel makinede bugun (kisa-pilot verisi ana veriyle birlestirilip
`build_features.py` ile) uretildi, GitHub'da DEGIL (data/ .gitignore'da). Asagidaki
4. hucrede bu ikisini yerel makineden yukleyeceksin:
- `data/processed/fused.parquet` (~44MB)
- `data/processed/holdout_ids.txt` (~58KB)

Kodun kendisi (measure_calibration.py + models/) zaten GitHub'da -- bu adim icin
yeni bir push gerekmiyor.

**Bu notebook'u Colab'a yukleme:** https://colab.research.google.com adresini ac,
sol ustten **File > Upload notebook** (veya **Dosya > Not defteri yukle**) sec,
bu dosyayi (`notebooks/colab_measure_calibration.ipynb`) bilgisayarindan sec.
GitHub'a push edilmesine gerek YOK -- notebook'un kendisi sadece hucrelerdeki
`git clone` komutuyla kodu ayrica cekiyor.

In [ ]:
# 1. GPU kontrolu
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# 2. Repo klonla (ilk kez) veya guncelle (klasor zaten varsa git pull)
GITHUB_REPO = 'https://github.com/BurakCANKURT/humanai-detect.git'
import os
if os.path.exists('/content/humanai_detect'):
    %cd /content/humanai_detect
    !git pull origin master
else:
    !git clone {GITHUB_REPO} /content/humanai_detect
    %cd /content/humanai_detect

In [ ]:
# 3. Bagimliliklari kur (birkac dakika surer)
!pip install -e '.' -q

In [ ]:
# 4. Guncel veri dosyalarini yukle (fused.parquet + holdout_ids.txt)
# Calistirinca acilan dosya secme penceresinden IKISINI BIRDEN sec:
#   data/processed/fused.parquet
#   data/processed/holdout_ids.txt
import os
os.makedirs('data/processed', exist_ok=True)
from google.colab import files
uploaded = files.upload()
import shutil
for fname in uploaded:
    shutil.move(fname, f'data/processed/{fname}')
print(os.listdir('data/processed'))

In [ ]:
# 5. Kalibrasyon olcumu (asil is)
!python scripts/measure_calibration.py --stage after --save-models

In [ ]:
# 6. Sonuclari yerel makineye indir
from google.colab import files
files.download('outputs/reports/cv_results/calibration_before_after_after.json')
files.download('outputs/models/_diag_after_calibrated.pkl')

## Indirilen Dosyalari Yerlestirme

Bu iki dosyayi indirdikten sonra (genelde `Downloads/` klasorune duser), Claude'a
haber ver -- dosya yolunu soylemen yeterli, `outputs/reports/cv_results/` ve
`outputs/models/` altina yerlestirip held-out sonuclarina bakacak ve ardindan
kisa-metin OOD tanisini tekrarlayacak.